# 04b — Inférence Seq2Point

Ce notebook charge les modèles Seq2Point entraînés (loss combinée) et génère les prédictions sur le jeu de test UK-DALE (maison 5).  
Les séries temporelles prédites sont sauvegardées pour être réutilisées dans les notebooks 06 (évaluation) et 08 (flexibilité).

**Pas d'entraînement ici** — lecture seule des poids `.pt`.

In [1]:
import json
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

ROOT      = Path("..")
DATA      = ROOT / "data/processed/seq2point"
MODELS    = ROOT / "models/combined"
NORM_FILE = ROOT / "data/processed/norm_params.json"
OUT       = ROOT / "outputs/seq2point"
OUT.mkdir(parents=True, exist_ok=True)

WINDOW  = 599
HALF    = WINDOW // 2   # 299 — indice du point central
STRIDE  = 20            # cohérent avec l'évaluation du notebook 04
BATCH   = 2048
DATASET = "UK-DALE"

with open(NORM_FILE) as f:
    NORM = json.load(f)

print("Dossier de sortie :", OUT)
print("Appareils disponibles :", list(NORM[DATASET]["appliances"].keys()))

Dossier de sortie : ..\outputs\seq2point
Appareils disponibles : ['fridge', 'washing machine', 'dish washer', 'microwave', 'kettle']


## Architecture Seq2Point

Reconstruction exacte à partir de l'inspection des clés du state-dict :  
5 × Conv1d (`padding='same'`) → Flatten → Linear(29 950, 1024) → 2 têtes (puissance + état ON/OFF).

In [2]:
class Seq2Point(nn.Module):
    def __init__(self, window=599):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,  30, 10, padding="same"), nn.ReLU(),
            nn.Conv1d(30, 30,  8, padding="same"), nn.ReLU(),
            nn.Conv1d(30, 40,  6, padding="same"), nn.ReLU(),
            nn.Conv1d(40, 50,  5, padding="same"), nn.ReLU(),
            nn.Conv1d(50, 50,  5, padding="same"), nn.ReLU(),
        )
        self.shared     = nn.Sequential(nn.Flatten(), nn.Linear(50 * window, 1024), nn.ReLU())
        self.head_power = nn.Linear(1024, 1)
        self.head_state = nn.Linear(1024, 1)

    def forward(self, x):
        h = self.shared(self.conv(x))
        return self.head_power(h).squeeze(-1), self.head_state(h).squeeze(-1)

# Vérification rapide du chargement
test_model = Seq2Point(WINDOW)
sd = torch.load(MODELS / "seq2point_fridge_UK-DALE.pt", map_location="cpu", weights_only=True)
test_model.load_state_dict(sd)
print("Modèle chargé, paramètres :", sum(p.numel() for p in test_model.parameters()), "params")

Modèle chargé, paramètres : 30709274 params


## Fonction d'inférence

Pour chaque segment continu (délimité par `offsets`), on crée des fenêtres glissantes avec `stride=20`.  
La prédiction correspond au **point central** de chaque fenêtre (indice `start + 299`).  
Les valeurs sont dénormalisées et écrêtées à 0 W (contrainte physique).

In [3]:
APP_TO_FILE = {
    "fridge":          "fridge",
    "kettle":          "kettle",
    "microwave":       "microwave",
    "dish washer":     "dish_washer",
    "washing machine": "washing_machine",
}

def run_inference(appliance, dataset=DATASET, stride=STRIDE):
    fname    = APP_TO_FILE[appliance]
    npz_path = DATA   / f"{fname}__{dataset}__test.npz"
    pt_path  = MODELS / f"seq2point_{fname}_{dataset}.pt"

    # Chargement des données
    d            = np.load(npz_path, allow_pickle=True)
    mains        = d["mains"].astype(np.float32)
    target       = d["target"].astype(np.float32)
    offsets      = d["offsets"].astype(np.int64)
    on_threshold = float(d["on_threshold"])

    # Paramètres de normalisation
    norm = NORM[dataset]
    mu_m, sd_m = norm["mains"]["mean"], norm["mains"]["std"]
    mu_a       = norm["appliances"][appliance]["mean"]
    sd_a       = norm["appliances"][appliance]["std"]

    # Construction de l'index des fenêtres : (debut_fenetre, indice_centre)
    pairs = []
    for i in range(len(offsets) - 1):
        s, e = int(offsets[i]), int(offsets[i + 1])
        n = e - s
        if n < WINDOW:
            continue
        for p in range(0, n - WINDOW + 1, stride):
            pairs.append((s + p, s + p + HALF))
    pairs = np.array(pairs, dtype=np.int64)  # (N, 2)

    # Chargement du modèle
    model = Seq2Point(WINDOW)
    model.load_state_dict(torch.load(pt_path, map_location="cpu", weights_only=True))
    model.eval()

    # Inférence par lots
    pred_norm = np.empty(len(pairs), dtype=np.float32)
    with torch.no_grad():
        for b in range(0, len(pairs), BATCH):
            starts = pairs[b : b + BATCH, 0]
            x = np.stack([(mains[s : s + WINDOW] - mu_m) / sd_m for s in starts])
            p_hat, _ = model(torch.from_numpy(x).unsqueeze(1))  # (B,1,W) → modele
            pred_norm[b : b + len(starts)] = p_hat.numpy()

    # Dénormalisation + écrêtage physique
    pred_watts = np.maximum(pred_norm * sd_a + mu_a, 0.0)
    gt_watts   = target[pairs[:, 1]]

    return {
        "pred_watts":     pred_watts,
        "gt_watts":       gt_watts,
        "center_indices": pairs[:, 1],
        "on_threshold":   on_threshold,
    }

print("Fonction d'inférence définie.")

Fonction d'inférence définie.


## Exécution sur les 5 appareils

In [4]:
results = {}
for app in APP_TO_FILE:
    print(f"  {app:<20} ...", end=" ", flush=True)
    results[app] = run_inference(app)
    r = results[app]
    mae = np.mean(np.abs(r["pred_watts"] - r["gt_watts"]))
    print(f"MAE={mae:6.1f} W   ({len(r['pred_watts']):>6} prédictions)")

print("\nInférence terminée.")

  fridge               ... 

c:\Users\acret\Documents\Ponts 2A\Semestre 2\Projet_scientifique\projet-nilm\.venv\Lib\site-packages\torch\nn\modules\conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1091.)
  return F.conv1d(


MAE=  24.5 W   ( 29248 prédictions)
  kettle               ... MAE=   9.2 W   ( 42838 prédictions)
  microwave            ... MAE=  51.6 W   ( 33883 prédictions)
  dish washer          ... MAE=  20.4 W   ( 33883 prédictions)
  washing machine      ... MAE=  31.4 W   ( 33883 prédictions)

Inférence terminée.


## Contrôle de cohérence

Comparaison des MAE recalculés avec les valeurs de référence de `metrics_combined.json`  
(légères différences possibles : seed, ordre des batches).

In [6]:
ref_path = ROOT / "outputs/exploration/metrics_combined.json"
with open(ref_path) as f:
    ref = json.load(f)

print(f"{'Appareil':<22} {'MAE recalculé (W)':>18} {'MAE référence (W)':>18} {'F1 recalculé':>13} {'F1 référence':>13}")
print("-" * 88)
for app in APP_TO_FILE:
    r   = results[app]
    thr = r["on_threshold"]
    mae = np.mean(np.abs(r["pred_watts"] - r["gt_watts"]))
    f1  = f1_score(r["gt_watts"] > thr, r["pred_watts"] > thr, zero_division=0)

    key      = f"{app}__UK-DALE"
    ref_mae  = ref.get(key, {}).get("mae_w", float("nan"))
    ref_f1   = ref.get(key, {}).get("f1",    float("nan"))
    print(f"{app:<22} {mae:>18.2f} {ref_mae:>18.2f} {f1:>13.3f} {ref_f1:>13.3f}")

Appareil                MAE recalculé (W)  MAE référence (W)  F1 recalculé  F1 référence
----------------------------------------------------------------------------------------
fridge                              24.49              24.49         0.826         0.826
kettle                               9.24               9.24         0.850         0.850
microwave                           51.61              51.61         0.026         0.026
dish washer                         20.41              20.41         0.367         0.367
washing machine                     31.41              31.41         0.346         0.346


## Sauvegarde

Fichier `predictions_combined_UK-DALE_test.npz` contenant pour chaque appareil :  
- `{app}__pred_watts` : prédictions dénormalisées (W)  
- `{app}__gt_watts` : vérité terrain (W)  
- `{app}__center_indices` : positions dans la séquence brute  
- `{app}__on_threshold` : seuil ON/OFF (W)

In [7]:
save_dict = {}
for app, r in results.items():
    save_dict[f"{app}__pred_watts"]     = r["pred_watts"]
    save_dict[f"{app}__gt_watts"]       = r["gt_watts"]
    save_dict[f"{app}__center_indices"] = r["center_indices"]
    save_dict[f"{app}__on_threshold"]   = np.array(r["on_threshold"])

out_path = OUT / "predictions_combined_UK-DALE_test.npz"
np.savez_compressed(out_path, **save_dict)
print("Sauvegardé :", out_path)
print("Clés :", list(save_dict.keys()))

Sauvegardé : ..\outputs\seq2point\predictions_combined_UK-DALE_test.npz
Clés : ['fridge__pred_watts', 'fridge__gt_watts', 'fridge__center_indices', 'fridge__on_threshold', 'kettle__pred_watts', 'kettle__gt_watts', 'kettle__center_indices', 'kettle__on_threshold', 'microwave__pred_watts', 'microwave__gt_watts', 'microwave__center_indices', 'microwave__on_threshold', 'dish washer__pred_watts', 'dish washer__gt_watts', 'dish washer__center_indices', 'dish washer__on_threshold', 'washing machine__pred_watts', 'washing machine__gt_watts', 'washing machine__center_indices', 'washing machine__on_threshold']
